Groundwater | Flow Modeling Track

# Step 4: Model Implementation – MODFLOW 6

> **The 10 steps at a glance:** 1-Goal → 2-Perceptual → 3-MODFLOW → **4-Build** → 5-Calibrate → 6-Validate → 7-Sensitivity → 8-Apply → 9-Document → 10-Communicate

**Story so far:** In Steps 1-3, you defined clear objectives, developed a perceptual model of the Limmat Valley aquifer, and learned MODFLOW 6 fundamentals. Now we **build the numerical model** – translating the perceptual understanding into a working FloPy simulation.

| **Core content** | ~50-55 minutes |
|:--|:--|
| **Optional: Additional details & exploration** | +10-15 minutes |

## Learning Objectives

By the end of this notebook, you will be able to:

1. **Implement** a groundwater flow model using FloPy and MODFLOW 6
2. **Create** unstructured (DISV) Voronoi grids with river alignment
3. **Apply** boundary conditions appropriate for the Limmat Valley
4. **Run** and verify a MODFLOW 6 simulation
5. **Interpret** model results and water balance

## Prerequisites

Before starting this notebook, you should have:
- **Completed [2_perceptual_model.ipynb](2_perceptual_model.ipynb)** – understanding of Limmat Valley hydrogeology
- **Completed [3_modflow_fundamentals.ipynb](3_modflow_fundamentals.ipynb)** – MODFLOW 6 packages and DISV concepts
- Familiarity with MODFLOW packages (NPF, CHD, RIV, WEL, RCH)
- Basic Python and FloPy experience

> **Where does data go?** Files downloaded in this notebook are saved to `~/applied_groundwater_modelling_data/`. The download functions print the exact path each time.

---

## Introduction

In this notebook, we build a **numerical groundwater flow model** for the Limmat Valley aquifer using MODFLOW 6 and FloPy. The model uses an unstructured Voronoi grid (DISV) which allows flexible local refinement for your case study work.

We follow the steps a professional groundwater modeler would take:
1. Set up the model grid with river-aligned cells (Chapters 1-2)
2. Define model geometry from DEM and aquifer thickness data (Chapter 3)
3. Assign hydraulic parameters (Chapter 4)
4. Apply boundary conditions: CHD, WEL, RIV, RCH (Chapter 5)
5. Run and verify the model (Chapters 6-7)

> **💡 Tip: Use the Outline Panel**
> This is a long notebook with many sections. To keep track of where you are:
> - **In VS Code**: Open the Outline panel (View → Open View → Outline)
> - **In JupyterHub/JupyterLab**: Open the Table of Contents panel (View → Table of Contents)

## 1 - Setup

First, we import the necessary libraries and set up the model workspace.

In [ ]:
# Setup - import required libraries
import sys
import os
import warnings
warnings.filterwarnings('ignore')

# Scientific computing
import numpy as np
import pandas as pd
import geopandas as gpd
import matplotlib.pyplot as plt

# FloPy - MODFLOW 6
import flopy
from flopy.mf6 import MFSimulation, ModflowTdis, ModflowIms
from flopy.mf6 import ModflowGwf, ModflowGwfdisv
from flopy.mf6 import ModflowGwfnpf, ModflowGwfic, ModflowGwfoc
from flopy.mf6 import ModflowGwfrcha, ModflowGwfchd, ModflowGwfwel, ModflowGwfriv

# Course utilities
sys.path.insert(0, '../../_SUPPORT/src')
sys.path.insert(0, '../../_SUPPORT/src/scripts/scripts_exercises')

from data_utils import download_named_file, get_default_data_folder
from disv_grid_utils import create_grid_with_rivers, export_grid_to_geopackage
from boundary_utils import assign_idomain_from_geometry
from model_io_utils import sample_raster_at_cells, format_budget_summary, load_and_interpolate_aquifer_thickness
from plot_utils import quick_model_plot, get_idomain_cmap
from plot_utils import plot_chd_validation

# Checkpoint utilities
from shared_functions import check_task_with_solution, check_dict_task_with_solution, create_multiple_choice

print(f"FloPy version: {flopy.__version__}")

In [ ]:
# Define workspace paths using os.path.join (consistent with other notebooks)
DATA_DIR = get_default_data_folder()
MODEL_WS = os.path.join(DATA_DIR, 'notebook4_model')

if not os.path.exists(MODEL_WS):
    os.makedirs(MODEL_WS)

print(f"Data directory: {DATA_DIR}")
print(f"Model workspace: {MODEL_WS}")

### 1.1 - Create MODFLOW 6 Simulation

MODFLOW 6 uses a simulation framework that can contain multiple models. We start by creating the simulation and time discretization.

In [ ]:
# Create a new MF6 simulation
sim_name = 'limmat_valley'
sim = MFSimulation(
    sim_name=sim_name,
    sim_ws=MODEL_WS,
    exe_name='mf6'  # MODFLOW 6 executable
)

# Time discretization - steady state (1 stress period)
tdis = ModflowTdis(
    sim,
    nper=1,  # Number of stress periods
    perioddata=[(1.0, 1, 1.0)]  # (perlen, nstp, tsmult)
)

print("Simulation created: steady-state, single stress period")

---

## 2 - Model Grid - DISV

We use a Voronoi-based unstructured grid (DISV = DISretization by Vertices). This grid type:
- Satisfies the control volume finite difference (CVFD) requirements
- Allows flexible local refinement (which you'll use in your case study)
- Is the modern approach for complex geometries

**River-Aligned Grid:** We incorporate river polygons as internal boundaries during grid generation. This ensures that:
- Voronoi cell edges align with river banks
- River-aquifer exchange calculations (RIV package) are more accurate
- Cells within/near rivers have finer resolution

**Recall from Notebook 3:** Voronoi cells have the property that cell centers are equidistant from all cell faces, ensuring flow directions are perpendicular to cell faces.

In [ ]:
# Load model boundary and river polygons
boundary_path = download_named_file(name='model_boundary', data_type='gis')
boundary_gdf = gpd.read_file(boundary_path)

rivers_path = download_named_file(name='rivers', data_type='gis')
rivers_gdf = gpd.read_file(rivers_path)

# Filter for Sihl and Limmat rivers (main surface water bodies)
rivers_gdf = rivers_gdf[rivers_gdf['GEWAESSERNAME'].isin(['Sihl', 'Limmat'])]

print(f"Model boundary CRS: {boundary_gdf.crs}")
print(f"Boundary area: {boundary_gdf.geometry.area.sum() / 1e6:.1f} km²")
print(f"Rivers loaded: {rivers_gdf['GEWAESSERNAME'].unique().tolist()}")

In [ ]:
# Create Voronoi grid with rivers as internal boundaries
# This ensures cells align with river banks for accurate RIV package calculations

cell_size = 50  # meters - target cell size for general domain
river_cell_size = 25  # meters - finer cells in/near rivers

grid_data = create_grid_with_rivers(
    boundary_gdf=boundary_gdf,
    river_gdf=rivers_gdf,
    cell_size=cell_size,
    river_cell_size=river_cell_size,
    crs=str(boundary_gdf.crs)
)

# Extract grid components
vor = grid_data['voronoi_grid']
modelgrid = grid_data['modelgrid']
gridprops = grid_data['disv_gridprops']
ncpl = gridprops['ncpl']  # Number of cells per layer
river_cells = grid_data['river_cells']

print(f"\nGrid created with {ncpl} cells")
print(f"Target cell size: {cell_size} m (general), {river_cell_size} m (rivers)")
print(f"River cells identified: {len(river_cells)}")

In [ ]:
# Visualize the grid with river cells highlighted
fig, ax = plt.subplots(figsize=(12, 10))

# Create array for river cell highlighting (0 = regular, 1 = river)
river_mask = np.zeros(ncpl)
if len(river_cells) > 0:
    river_mask[river_cells] = 1

# Use FloPy's PlotMapView for unstructured grid visualization
from flopy.plot import PlotMapView
pmv = PlotMapView(modelgrid=modelgrid, ax=ax)

# Plot grid with river cells colored
colors = pmv.plot_array(river_mask, cmap='Blues', alpha=0.6)
pmv.plot_grid(linewidth=0.3, color='gray')

# Add boundary and rivers
boundary_gdf.boundary.plot(ax=ax, color='red', linewidth=2, label='Model boundary')
rivers_gdf.boundary.plot(ax=ax, color='blue', linewidth=1.5, label='River banks')

ax.set_title(f'River-Aligned Voronoi Grid ({ncpl} cells)\n'
             f'General: ~{cell_size}m, Rivers: ~{river_cell_size}m')
ax.set_xlabel('Easting (m)')
ax.set_ylabel('Northing (m)')
ax.legend(loc='upper right')
ax.set_aspect('equal')
plt.tight_layout()
plt.show()

> **💡 Quick Check**
>
> Without looking at the code, can you explain why river polygons were passed to `create_grid_with_rivers()`?

<details>
<summary>Click to check your answer</summary>

River polygons create internal boundaries that force Voronoi cell edges to align with river banks, improving accuracy of river-aquifer exchange calculations in the RIV package.
</details>

### 2.1 - Numerical Checkpoint 1: Model Discretization

In [ ]:
# Check your grid cell count
check_task_with_solution('task04_checkpoint_1')

---

## 3 - Model Geometry

Now we define the vertical geometry of the model:
- **Model top**: Land surface elevation from DEM
- **Model bottom**: Computed from aquifer thickness data

In [ ]:
# Load DEM and sample at cell centers
dem_path = download_named_file(name='dem_hres', data_type='gis')

# Sample DEM at cell centers using FloPy's Raster utility
model_top = sample_raster_at_cells(dem_path, modelgrid)

print(f"Model top elevation range: {model_top.min():.1f} to {model_top.max():.1f} m a.s.l.")

In [ ]:
# Aquifer thickness interpolation
# The aquifer thickness varies spatially based on geological survey data.
# We interpolate from thickness contour lines (GS_GW_MAECHTIGKEIT_L layer)
# to our model grid. Uses two-stage interpolation:
# 1. Linear interpolation for smooth interior surfaces
# 2. Nearest-neighbor gap filling for areas outside the convex hull

# Download thickness data (same file as groundwater map)
gw_map_path = download_named_file(name='groundwater_map_norm', data_type='gis')

# Minimum aquifer thickness - used where interpolation data is sparse
# Based on geological knowledge of the Limmat Valley alluvial aquifer
min_aquifer_thickness = 10.0  # meters

# Interpolate thickness at grid cell centers
aquifer_thickness = load_and_interpolate_aquifer_thickness(
    gw_map_path=gw_map_path,
    modelgrid=modelgrid,
    boundary_gdf=boundary_gdf,
    method='robust',
    min_thickness=min_aquifer_thickness
)

# Apply minimum thickness constraint
aquifer_thickness = np.maximum(aquifer_thickness, min_aquifer_thickness)

# Compute model bottom elevation
model_bottom = model_top - aquifer_thickness

print(f"\nModel geometry summary:")
print(f"  Model top: {model_top.min():.1f} to {model_top.max():.1f} m a.s.l.")
print(f"  Thickness: {aquifer_thickness.min():.1f} to {aquifer_thickness.max():.1f} m")
print(f"  Minimum thickness constraint: {min_aquifer_thickness} m")
print(f"  Model bottom: {model_bottom.min():.1f} to {model_bottom.max():.1f} m a.s.l.")

In [ ]:
# Visualize model geometry
fig, axes = plt.subplots(1, 2, figsize=(14, 6))

quick_model_plot(modelgrid, model_top, boundary_gdf=boundary_gdf,
                 title='Model Top (Land Surface)', cmap='terrain',
                 colorbar_label='Elevation (m a.s.l.)', ax=axes[0])

quick_model_plot(modelgrid, aquifer_thickness, boundary_gdf=boundary_gdf,
                 title='Aquifer Thickness', cmap='Blues',
                 colorbar_label='Thickness (m)', ax=axes[1])

plt.tight_layout()
plt.show()

### 3.1 - Numerical Checkpoint 2: Aquifer Properties

In [ ]:
check_task_with_solution('task04_checkpoint_2')

---

## 4 - Hydraulic Parameters

The Limmat Valley aquifer is divided into three zones based on geological characteristics:

| Zone | Location | Sediment Type | K Range (m/day) |
|------|----------|---------------|----------------|
| 1 | Upstream | Sandy gravels | 15-25 |
| 2 | Downstream | Finer alluvium | 8-15 |
| 3 | Hardhof | Coarse gravels | 25-40 |

These ranges are based on field measurements (pumping tests) and Swiss cantonal guidelines (AWA Zurich).

### 4.1 - ✏️ Exercise: Hydraulic Conductivity Assignment

**Part A: Recall** (1 min)

From Notebook 2, what K range did we identify for Limmat Valley gravels?

<details>
<summary>Hint</summary>

Based on pumping tests and Swiss guidelines, K ranges from 10-40 m/day depending on sediment type.
</details>

**Part B: Zone Assignment** (2 min)

Using the guidance ranges in the table above, assign K values to each zone in the code cell below.

**Part C: Reflection** (1 min)

Which zone has the highest uncertainty in K? Why might this matter for calibration?

<details>
<summary>Hint</summary>

Zones with fewer pumping test measurements have higher uncertainty. The Hardhof zone (Zone 3) has the widest K range, suggesting more variability or fewer constraints.
</details>

In [ ]:
# EXERCISE: Fill in K values based on the guidance ranges above
k_values = {
    "zone_1_upstream": 20,      # Sandy gravels: 15-25 m/day
    "zone_2_downstream": 12,    # Finer alluvium: 8-15 m/day
    "zone_3_hardhof": 32,       # Coarse gravels: 25-40 m/day
}

# Check your values
check_dict_task_with_solution('task04_k_values', k_values)

In [ ]:
# For this base model, we use a uniform K value
# Zone-based assignment will be implemented in case study notebooks
k_array = np.full(ncpl, k_values['zone_1_upstream'])

print(f"K array shape: {k_array.shape}")
print(f"K value: {k_array[0]:.1f} m/day (uniform for base model)")

> **📘 Note on K Zones**
>
> In this base model, we use uniform K for simplicity. The zone-specific values you defined above will be used in the calibration notebook (Notebook 5) where we optimize K values by zone. For now, this establishes the workflow.

---

With grid geometry and hydraulic properties defined, we now specify how water enters and leaves the model domain. The boundary conditions translate the fluxes from your perceptual model (Notebook 2) into MODFLOW inputs.

## 5 - Boundary Conditions

We define the boundary conditions that control water flow:

1. **IDOMAIN**: Active/inactive cells
2. **RIV**: River-aquifer interaction (Sihl & Limmat)
3. **CHD**: Constant head at outflow (west) boundary
4. **WEL**: Lateral inflow from valley margins (north & south)
5. **RCH**: Areal recharge from precipitation

The east boundary is impermeable bedrock (no-flow, default).

### 5.1 Active Domain (IDOMAIN)

In [ ]:
# Create IDOMAIN - all cells within boundary are active
idomain = assign_idomain_from_geometry(boundary_gdf, modelgrid, nlay=1)
n_active = np.sum(idomain > 0)
print(f"Active cells: {n_active} of {ncpl} ({100*n_active/ncpl:.1f}%)")

### 5.2 River Boundary (RIV)

The Sihl and Limmat rivers exchange water with the aquifer based on the head difference between the river stage and the aquifer head. The RIV package requires:

- **Stage**: Water level in the river (from gauge measurements or DEM)
- **Conductance**: Controls exchange rate, calculated as:
  $$C = K_{riverbed} \times \frac{W \times L}{M}$$
  where $K_{riverbed}$ = riverbed hydraulic conductivity, $W$ = river width, $L$ = reach length, $M$ = riverbed thickness
- **Riverbed bottom (rbot)**: Elevation of the riverbed, extracted from high-resolution DEM

**River-specific parameters** are used because the Sihl and Limmat have different characteristics:
- **Sihl**: Narrower (15m), shallower (0.3m depth), lower riverbed K
- **Limmat**: Wider (30m), deeper (0.7m depth), higher riverbed K

The `river_cells` identified during grid creation are used to assign RIV boundaries.

In [ ]:
# River-aquifer interaction with river-specific parameters
# Get cell centers for spatial operations
xc = modelgrid.xcellcenters
yc = modelgrid.ycellcenters
if hasattr(xc, 'flatten'):
    xc = xc.flatten()
    yc = yc.flatten()

# Use river_cells identified during grid creation
if len(river_cells) > 0:
    # River-specific parameters (from perceptual model analysis)
    # Leakage coefficients estimated in perceptual model (converted from 1/s to 1/day)
    q_riv_sihl = 1.3e-6 * 86400    # 1/day (leakage coefficient)
    q_riv_limmat = 3.5e-6 * 86400  # 1/day (leakage coefficient)
    
    # Riverbed properties
    riverbed_thickness = 0.5  # m, typical assumption for riverbed sediment layer
    
    # Riverbed hydraulic conductivity: K = leakage_coeff × thickness
    riverbed_k_sihl = q_riv_sihl * riverbed_thickness      # ~0.056 m/day
    riverbed_k_limmat = q_riv_limmat * riverbed_thickness  # ~0.151 m/day
    
    # Average river depths from field observations
    sihl_depth_mean = 0.3   # m (Sihl is shallower, faster flowing)
    limmat_depth_mean = 0.7  # m (Limmat is deeper, slower flowing)
    
    # River widths
    width_sihl = 15.0    # m
    width_limmat = 30.0  # m
    
    # Minimum clearance between stage and rbot
    min_stage_clearance = 0.05  # m
    
    # Cross-section bin width for rbot homogenization
    bin_width = 100.0  # meters
    
    print("River parameters:")
    print(f"  Sihl:   K_riverbed = {riverbed_k_sihl:.3f} m/day, width = {width_sihl} m, depth = {sihl_depth_mean} m")
    print(f"  Limmat: K_riverbed = {riverbed_k_limmat:.3f} m/day, width = {width_limmat} m, depth = {limmat_depth_mean} m")
    print(f"  Riverbed thickness: {riverbed_thickness} m")
    print(f"  Cross-section bin width: {bin_width} m")
    
    # Determine which river each cell belongs to
    from shapely.geometry import Point, Polygon
    
    # Separate river geometries
    sihl_geom = rivers_gdf[rivers_gdf['GEWAESSERNAME'] == 'Sihl'].union_all()
    limmat_geom = rivers_gdf[rivers_gdf['GEWAESSERNAME'] == 'Limmat'].union_all()
    
    # Helper function to get cell area from vertex connectivity
    def get_cell_area(modelgrid, cell_id):
        """Calculate cell area from vertex geometry."""
        cell_data = modelgrid.cell2d[cell_id]
        nvert = cell_data[3]
        vert_ids = cell_data[4:4+nvert]
        vertices = modelgrid._vertices
        vert_dict = {v[0]: (v[1], v[2]) for v in vertices}
        coords = [vert_dict[vid] for vid in vert_ids]
        return Polygon(coords).area
    
    # =========================================================================
    # STEP 1: Classify cells by river and compute initial (raw) rbot values
    # =========================================================================
    cell_data = []  # Store cell info for processing
    
    for cell_id in river_cells:
        if idomain[0, cell_id] <= 0:
            continue
            
        cell_x = xc[cell_id]
        cell_y = yc[cell_id]
        cell_point = Point(cell_x, cell_y)
        
        # Classify by river
        if sihl_geom is not None and sihl_geom.contains(cell_point):
            river_name = 'Sihl'
            river_depth = sihl_depth_mean
            riverbed_k = riverbed_k_sihl
            river_width = width_sihl
        elif limmat_geom is not None and limmat_geom.contains(cell_point):
            river_name = 'Limmat'
            river_depth = limmat_depth_mean
            riverbed_k = riverbed_k_limmat
            river_width = width_limmat
        else:
            river_name = 'Unknown'
            river_depth = limmat_depth_mean
            riverbed_k = riverbed_k_limmat
            river_width = width_limmat
        
        # Compute raw rbot from DEM
        raw_rbot = model_top[cell_id] - river_depth - 0.5
        
        # Ensure rbot is above aquifer bottom
        if raw_rbot < model_bottom[cell_id]:
            raw_rbot = model_bottom[cell_id] + 0.1
        
        cell_data.append({
            'cell_id': cell_id,
            'x': cell_x,
            'y': cell_y,
            'river': river_name,
            'river_depth': river_depth,
            'riverbed_k': riverbed_k,
            'river_width': river_width,
            'raw_rbot': raw_rbot,
        })
    
    # Convert to arrays for easier processing
    import pandas as pd
    cell_df = pd.DataFrame(cell_data)
    
    print(f"\nClassified {len(cell_df)} active river cells:")
    print(f"  Sihl: {(cell_df['river'] == 'Sihl').sum()}")
    print(f"  Limmat: {(cell_df['river'] == 'Limmat').sum()}")
    print(f"  Unknown: {(cell_df['river'] == 'Unknown').sum()}")
    
    # =========================================================================
    # STEP 2: Bin cells into cross-sections and compute minimum rbot per bin
    # =========================================================================
    # Sihl flows roughly N-S, so bin by y-coordinate (perpendicular cross-sections)
    # Limmat flows roughly W, so bin by x-coordinate (perpendicular cross-sections)
    
    def assign_bins(df, coord_col, bin_width):
        """Assign bin indices based on coordinate."""
        coord_min = df[coord_col].min()
        bins = ((df[coord_col] - coord_min) // bin_width).astype(int)
        return bins
    
    # Assign bins separately for each river
    cell_df['bin'] = -1
    cell_df['homogenized_rbot'] = cell_df['raw_rbot']  # Default to raw
    
    # Process Sihl (bin by y-coordinate)
    sihl_mask = cell_df['river'] == 'Sihl'
    if sihl_mask.sum() > 0:
        sihl_bins = assign_bins(cell_df.loc[sihl_mask], 'y', bin_width)
        cell_df.loc[sihl_mask, 'bin'] = sihl_bins
        
        # Compute minimum rbot per Sihl bin
        for bin_id in sihl_bins.unique():
            bin_mask = sihl_mask & (cell_df['bin'] == bin_id)
            min_rbot = cell_df.loc[bin_mask, 'raw_rbot'].min()
            cell_df.loc[bin_mask, 'homogenized_rbot'] = min_rbot
        
        n_sihl_bins = sihl_bins.nunique()
        print(f"\nSihl: {n_sihl_bins} cross-section bins (by y-coordinate)")
    
    # Process Limmat (bin by x-coordinate)
    limmat_mask = cell_df['river'] == 'Limmat'
    if limmat_mask.sum() > 0:
        limmat_bins = assign_bins(cell_df.loc[limmat_mask], 'x', bin_width)
        # Offset bin IDs to avoid collision with Sihl bins
        limmat_bins = limmat_bins + 10000
        cell_df.loc[limmat_mask, 'bin'] = limmat_bins
        
        # Compute minimum rbot per Limmat bin
        for bin_id in limmat_bins.unique():
            bin_mask = limmat_mask & (cell_df['bin'] == bin_id)
            min_rbot = cell_df.loc[bin_mask, 'raw_rbot'].min()
            cell_df.loc[bin_mask, 'homogenized_rbot'] = min_rbot
        
        n_limmat_bins = limmat_bins.nunique()
        print(f"Limmat: {n_limmat_bins} cross-section bins (by x-coordinate)")
    
    # Process Unknown cells (use raw rbot, no homogenization)
    unknown_mask = cell_df['river'] == 'Unknown'
    if unknown_mask.sum() > 0:
        print(f"Unknown: {unknown_mask.sum()} cells (no homogenization)")
    
    # Report homogenization effect
    rbot_diff = cell_df['raw_rbot'] - cell_df['homogenized_rbot']
    print(f"\nRbot homogenization effect:")
    print(f"  Max adjustment: {rbot_diff.max():.2f} m")
    print(f"  Mean adjustment: {rbot_diff.mean():.2f} m")
    print(f"  Cells adjusted: {(rbot_diff > 0.01).sum()} of {len(cell_df)}")
    
    # =========================================================================
    # STEP 3: Build RIV stress period data with homogenized rbot
    # =========================================================================
    riv_spd = []
    
    for _, row in cell_df.iterrows():
        cell_id = row['cell_id']
        rbot = row['homogenized_rbot']
        river_depth = row['river_depth']
        riverbed_k = row['riverbed_k']
        river_width = row['river_width']
        
        # Stage: water surface elevation
        stage = rbot + river_depth
        
        # Ensure stage is below land surface
        if stage >= model_top[cell_id]:
            stage = model_top[cell_id] - 0.1
            # Don't adjust rbot here - keep homogenized value
        
        # Ensure minimum clearance
        if stage - rbot < min_stage_clearance:
            stage = rbot + min_stage_clearance
        
        # Conductance calculation: C = K × W × L / M
        cell_area = get_cell_area(modelgrid, cell_id)
        reach_length = np.sqrt(cell_area)
        conductance = riverbed_k * river_width * reach_length / riverbed_thickness
        
        riv_spd.append([(0, cell_id), stage, conductance, rbot])
    
    # Summary statistics
    n_sihl = (cell_df['river'] == 'Sihl').sum()
    n_limmat = (cell_df['river'] == 'Limmat').sum()
    n_unknown = (cell_df['river'] == 'Unknown').sum()
    
    print(f"\nRIV package summary:")
    print(f"  Total river cells: {len(riv_spd)}")
    print(f"    Sihl cells: {n_sihl}")
    print(f"    Limmat cells: {n_limmat}")
    print(f"    Unassigned: {n_unknown}")
    
    if len(riv_spd) > 0:
        stages = [r[1] for r in riv_spd]
        conds = [r[2] for r in riv_spd]
        rbots = [r[3] for r in riv_spd]
        print(f"  Stage range: {min(stages):.1f} to {max(stages):.1f} m a.s.l.")
        print(f"  Conductance range: {min(conds):.1f} to {max(conds):.1f} m²/day")
        print(f"  Rbot range (homogenized): {min(rbots):.1f} to {max(rbots):.1f} m a.s.l.")
else:
    riv_spd = []
    print("No river cells identified - RIV package will not be created")

In [ ]:
# Visualize river boundary: cross-sections at 3 representative locations
# Also show the effect of rbot homogenization

if len(riv_spd) > 0:
    # Build lookup arrays from riv_spd
    riv_cell_ids = np.array([r[0][1] for r in riv_spd])
    riv_stages = np.array([r[1] for r in riv_spd])
    riv_conds = np.array([r[2] for r in riv_spd])
    riv_rbots = np.array([r[3] for r in riv_spd])
    
    # Get coordinates for river cells
    riv_x = xc[riv_cell_ids]
    riv_y = yc[riv_cell_ids]
    
    # Get raw rbot values from cell_df for comparison
    raw_rbots = cell_df.set_index('cell_id').loc[riv_cell_ids, 'raw_rbot'].values
    
    # Identify Sihl vs Limmat cells
    sihl_mask = np.array([sihl_geom.contains(Point(riv_x[i], riv_y[i])) for i in range(len(riv_cell_ids))])
    limmat_mask = np.array([limmat_geom.contains(Point(riv_x[i], riv_y[i])) for i in range(len(riv_cell_ids))])
    
    # Select representative cells (one from each region)
    sihl_indices = np.where(sihl_mask)[0]
    limmat_indices = np.where(limmat_mask)[0]
    
    sihl_mid_idx = sihl_indices[np.argmin(np.abs(riv_y[sihl_indices] - np.median(riv_y[sihl_indices])))] if len(sihl_indices) > 0 else None
    limmat_upstream_idx = limmat_indices[np.argmax(riv_y[limmat_indices])] if len(limmat_indices) > 0 else None
    limmat_downstream_idx = limmat_indices[np.argmin(riv_y[limmat_indices])] if len(limmat_indices) > 0 else None
    
    # Create figure with 2 rows
    fig = plt.figure(figsize=(14, 10))
    gs = fig.add_gridspec(2, 3, height_ratios=[1.2, 1], hspace=0.3, wspace=0.3)
    
    # Top row: Map showing rbot homogenization effect
    ax_map = fig.add_subplot(gs[0, :])
    
    # Plot river cells colored by rbot adjustment (raw - homogenized)
    rbot_adjustment = raw_rbots - riv_rbots
    scatter = ax_map.scatter(riv_x, riv_y, c=rbot_adjustment, cmap='RdYlBu_r', 
                             s=8, alpha=0.7, vmin=0, vmax=max(0.5, rbot_adjustment.max()))
    cbar = plt.colorbar(scatter, ax=ax_map, shrink=0.6, pad=0.02)
    cbar.set_label('Rbot adjustment (m)', fontsize=10)
    
    # Mark the 3 selected locations
    markers = []
    labels = []
    
    if sihl_mid_idx is not None:
        m1 = ax_map.scatter(riv_x[sihl_mid_idx], riv_y[sihl_mid_idx], 
                           c='red', s=200, marker='*', edgecolors='black', linewidths=1.5, zorder=10)
        markers.append(m1)
        labels.append('A: Sihl')
    
    if limmat_upstream_idx is not None:
        m2 = ax_map.scatter(riv_x[limmat_upstream_idx], riv_y[limmat_upstream_idx],
                           c='blue', s=200, marker='*', edgecolors='black', linewidths=1.5, zorder=10)
        markers.append(m2)
        labels.append('B: Limmat (upstream)')
    
    if limmat_downstream_idx is not None:
        m3 = ax_map.scatter(riv_x[limmat_downstream_idx], riv_y[limmat_downstream_idx],
                           c='green', s=200, marker='*', edgecolors='black', linewidths=1.5, zorder=10)
        markers.append(m3)
        labels.append('C: Limmat (downstream)')
    
    boundary_gdf.boundary.plot(ax=ax_map, color='black', linewidth=1.5)
    ax_map.set_xlabel('Easting (m)')
    ax_map.set_ylabel('Northing (m)')
    ax_map.set_title(f'River Bottom Homogenization Effect\n(bin width: {bin_width}m, blue=no change, red=lowered)')
    ax_map.legend(markers, labels, loc='upper right')
    ax_map.set_aspect('equal')
    
    # Bottom row: Cross-section profiles at each location
    sample_points = [
        (sihl_mid_idx, 'A: Sihl', 'red'),
        (limmat_upstream_idx, 'B: Limmat (upstream)', 'blue'),
        (limmat_downstream_idx, 'C: Limmat (downstream)', 'green')
    ]
    
    for i, (idx, label, color) in enumerate(sample_points):
        ax = fig.add_subplot(gs[1, i])
        
        if idx is not None:
            cell_id = riv_cell_ids[idx]
            top_val = model_top[cell_id]
            bot_val = model_bottom[cell_id]
            stage_val = riv_stages[idx]
            rbot_val = riv_rbots[idx]  # Homogenized
            raw_rbot_val = raw_rbots[idx]  # Original
            
            # Create profile bars
            x_pos = [0, 1, 2, 3, 4]
            heights = [top_val, stage_val, raw_rbot_val, rbot_val, bot_val]
            colors_bar = ['saddlebrown', 'deepskyblue', 'lightcoral', 'darkblue', 'gray']
            bar_labels = ['Model\nTop', 'River\nStage', 'Raw\nRbot', 'Homog.\nRbot', 'Model\nBottom']
            
            bars = ax.bar(x_pos, heights, color=colors_bar, edgecolor='black', width=0.7)
            
            # Add value labels on bars
            for bar, h in zip(bars, heights):
                ax.text(bar.get_x() + bar.get_width()/2, h + 0.3, f'{h:.1f}',
                       ha='center', va='bottom', fontsize=8)
            
            ax.set_xticks(x_pos)
            ax.set_xticklabels(bar_labels, fontsize=8)
            ax.set_ylabel('Elevation (m a.s.l.)')
            ax.set_title(f'{label}\nCell {cell_id}', color=color, fontweight='bold')
            
            # Set y-axis
            y_min = bot_val - 3
            y_max = top_val + 3
            ax.set_ylim(y_min, y_max)
            ax.grid(True, axis='y', alpha=0.3)
            
            # Add annotation
            adjustment = raw_rbot_val - rbot_val
            ax.text(0.02, 0.02, f'Rbot lowered by: {adjustment:.2f}m',
                   transform=ax.transAxes, fontsize=8, verticalalignment='bottom',
                   bbox=dict(boxstyle='round', facecolor='wheat', alpha=0.8))
        else:
            ax.text(0.5, 0.5, 'No data', ha='center', va='center', transform=ax.transAxes)
            ax.set_title(label)
    
    plt.suptitle('River Boundary Cross-Sections (with Homogenization)', fontsize=14, fontweight='bold', y=1.02)
    plt.tight_layout()
    plt.show()
    
    # Print summary
    print("\nCross-section summary:")
    for idx, label, _ in sample_points:
        if idx is not None:
            cell_id = riv_cell_ids[idx]
            adj = raw_rbots[idx] - riv_rbots[idx]
            print(f"  {label}: cell {cell_id}, rbot adjustment: {adj:.2f}m")

### 5.3 Outflow Boundary (CHD)

The western boundary represents the valley outlet where groundwater leaves the model domain. We implement this as a Constant Head (CHD) boundary condition.

| Boundary | Location | Package | Description |
|----------|----------|---------|-------------|
| **Outflow** | West | CHD | Fixed head at ~390 m a.s.l. (valley outlet) |

The CHD head value is estimated from groundwater isoline maps at the valley outlet.

In [ ]:
# CHD at outflow boundary (west)
# Load boundary segments to identify outflow cells
segments_path = download_named_file(name='model_boundary_segments', data_type='gis')
boundary_segments = gpd.read_file(segments_path)

# Create cell center points for spatial join
from shapely.geometry import Point
cell_points = gpd.GeoDataFrame(
    {'cell_id': range(ncpl)},
    geometry=[Point(x, y) for x, y in zip(xc, yc)],
    crs=boundary_gdf.crs
)

# Get outflow boundary (west) and buffer slightly to capture nearby cells
outflow_line = boundary_segments[boundary_segments['desc'] == 'outflow'].geometry.values[0]
outflow_buffer = outflow_line.buffer(75)  # 75m buffer to capture cells near boundary

# Find cells within buffer of outflow boundary
outflow_cells = cell_points[cell_points.geometry.within(outflow_buffer)]
outflow_cell_ids = outflow_cells['cell_id'].tolist()

# Filter to only active cells
outflow_cell_ids = [i for i in outflow_cell_ids if idomain[0, i] > 0]

# CHD head at outflow (downstream) - from groundwater isoline map
chd_head_outflow = 390.0  # m a.s.l.

# Verify heads are below land surface
outflow_min_top = model_top[outflow_cell_ids].min()
if chd_head_outflow >= outflow_min_top:
    chd_head_outflow = outflow_min_top - 1.0
    print(f"Warning: Adjusted CHD head to {chd_head_outflow:.1f} m (below land surface)")

# Create CHD stress period data
chd_spd = [[(0, i), chd_head_outflow] for i in outflow_cell_ids]

print(f"CHD boundary (outflow/west):")
print(f"  Number of cells: {len(chd_spd)}")
print(f"  Head value: {chd_head_outflow:.1f} m a.s.l.")
print(f"  Land surface range: {model_top[outflow_cell_ids].min():.1f} to {model_top[outflow_cell_ids].max():.1f} m")

# Validation figure
fig, axes = plot_chd_validation(
    chd_spd=chd_spd,
    modelgrid=modelgrid,
    model_top=model_top,
    model_bottom=model_bottom,
    idomain=idomain,
    boundary_gdf=boundary_gdf,
    riv_spd=riv_spd,
    buffer_m=100,
    title="CHD Boundary Validation - Outflow (West)"
)
plt.show()

### 5.4 Lateral Inflow (WEL)

Groundwater enters the Limmat Valley aquifer from the north and south valley margins. We represent this as injection wells along these boundaries.

**Boundary Cell Identification:** We identify cells along the north and south edges of the active domain using the boundary segment geometries. Cells within a buffer distance of these segments receive the lateral inflow.

The lateral inflow rate is estimated using a **water balance approach** based on:
- Catchment areas of adjacent hillslopes (north: ~11 km², south: ~15 km²)
- Annual precipitation (~1100 mm/year)
- Fraction of precipitation that becomes groundwater recharge to the valley (~10%)

This approach estimates how much water infiltrates on the surrounding hillslopes and flows toward the valley aquifer as lateral groundwater inflow.

In [ ]:
# Lateral inflow from valley margins using water balance approach
# Strategy: Use boundary segment geometry + edge detection to identify boundary cells
# 1. Buffer the north/south boundary segments to get candidate cells
# 2. Apply edge detection within candidates to get exactly one row of cells

from shapely.geometry import Point
import pandas as pd

# Load boundary segments (north and south inflow boundaries)
# boundary_segments was loaded earlier for CHD
north_line = boundary_segments[boundary_segments['desc'] == 'north'].union_all()
south_line = boundary_segments[boundary_segments['desc'] == 'south'].union_all()

print(f"Boundary segments loaded:")
print(f"  North: {north_line.geom_type}, length: {north_line.length:.0f} m")
print(f"  South: {south_line.geom_type}, length: {south_line.length:.0f} m")

# Create DataFrame of all active cells with coordinates
active_cells = []
for cell_id in range(ncpl):
    if idomain[0, cell_id] > 0:
        active_cells.append({
            'cell_id': cell_id,
            'x': xc[cell_id],
            'y': yc[cell_id],
            'point': Point(xc[cell_id], yc[cell_id])
        })

cells_df = pd.DataFrame(active_cells)

# Buffer distance for candidate cell selection (adaptive to cell size)
# Use a larger buffer to ensure we capture all boundary-adjacent cells
buffer_distance = 150  # meters

# Create buffers around boundary segments
north_buffer = north_line.buffer(buffer_distance)
south_buffer = south_line.buffer(buffer_distance)

# Identify candidate cells near each boundary
cells_df['near_north'] = cells_df['point'].apply(lambda p: north_buffer.contains(p))
cells_df['near_south'] = cells_df['point'].apply(lambda p: south_buffer.contains(p))

north_candidates = cells_df[cells_df['near_north']].copy()
south_candidates = cells_df[cells_df['near_south']].copy()

print(f"\nCandidate cells within {buffer_distance}m buffer:")
print(f"  North: {len(north_candidates)} candidates")
print(f"  South: {len(south_candidates)} candidates")

# Apply edge detection within candidates: for each x-bin, select outermost cell
x_bin_width = 50  # meters

# North boundary: select cell with highest y (most northern) in each x-bin
north_cell_ids = []
if len(north_candidates) > 0:
    north_candidates['x_bin'] = ((north_candidates['x'] - north_candidates['x'].min()) // x_bin_width).astype(int)
    for x_bin, group in north_candidates.groupby('x_bin'):
        north_idx = group['y'].idxmax()  # Highest y = northernmost
        north_cell_ids.append(group.loc[north_idx, 'cell_id'])
    north_cell_ids = list(set(north_cell_ids))

# South boundary: select cell with lowest y (most southern) in each x-bin
south_cell_ids = []
if len(south_candidates) > 0:
    south_candidates['x_bin'] = ((south_candidates['x'] - south_candidates['x'].min()) // x_bin_width).astype(int)
    for x_bin, group in south_candidates.groupby('x_bin'):
        south_idx = group['y'].idxmin()  # Lowest y = southernmost
        south_cell_ids.append(group.loc[south_idx, 'cell_id'])
    south_cell_ids = list(set(south_cell_ids))

# Remove any cells that might be in both lists (shouldn't happen with proper geometry)
south_cell_ids = [c for c in south_cell_ids if c not in north_cell_ids]

print(f"\nEdge detection within candidates (x-bin width: {x_bin_width}m):")
print(f"  North boundary: {len(north_cell_ids)} cells (northernmost per x-bin)")
print(f"  South boundary: {len(south_cell_ids)} cells (southernmost per x-bin)")

# Water balance approach for lateral inflow estimation
# Parameters from perceptual model (Notebook 2):
catchment_north_m2 = 11_000_000  # 11 km²
catchment_south_m2 = 15_000_000  # 15 km²
effective_recharge_m_per_year = 0.11  # 10% of 1100 mm/year

# Calculate total lateral inflow (m³/day)
Q_north_total = catchment_north_m2 * effective_recharge_m_per_year / 365.25
Q_south_total = catchment_south_m2 * effective_recharge_m_per_year / 365.25

print(f"\nWater balance lateral inflow estimation:")
print(f"  North catchment: {catchment_north_m2/1e6:.0f} km²")
print(f"  South catchment: {catchment_south_m2/1e6:.0f} km²")
print(f"  Effective recharge: {effective_recharge_m_per_year*1000:.0f} mm/year")
print(f"\nEstimated total lateral inflow:")
print(f"  North: {Q_north_total:.0f} m³/day")
print(f"  South: {Q_south_total:.0f} m³/day")

# Create WEL stress period data - distribute inflow evenly among boundary cells
wel_spd = []

if len(north_cell_ids) > 0:
    q_per_cell_north = Q_north_total / len(north_cell_ids)
    for cell_id in north_cell_ids:
        wel_spd.append([(0, cell_id), q_per_cell_north])
    print(f"\n  North: {len(north_cell_ids)} cells, Q per cell: {q_per_cell_north:.1f} m³/day")
else:
    q_per_cell_north = 0
    print("\n  Warning: No north boundary cells identified")

if len(south_cell_ids) > 0:
    q_per_cell_south = Q_south_total / len(south_cell_ids)
    for cell_id in south_cell_ids:
        wel_spd.append([(0, cell_id), q_per_cell_south])
    print(f"  South: {len(south_cell_ids)} cells, Q per cell: {q_per_cell_south:.1f} m³/day")
else:
    q_per_cell_south = 0
    print("  Warning: No south boundary cells identified")

print(f"\nTotal WEL cells: {len(wel_spd)}")
print(f"Total lateral inflow: {sum([w[1] for w in wel_spd]):.0f} m³/day")

In [ ]:
# Visualize lateral inflow (WEL) boundary conditions - Interactive map
import folium
from pyproj import Transformer

if len(wel_spd) > 0:
    # Transform coordinates from Swiss LV95 (EPSG:2056) to WGS84 (EPSG:4326) for Folium
    transformer = Transformer.from_crs("EPSG:2056", "EPSG:4326", always_xy=True)
    
    # Transform WEL cell coordinates
    north_coords_wgs84 = [transformer.transform(xc[cid], yc[cid]) for cid in north_cell_ids]
    south_coords_wgs84 = [transformer.transform(xc[cid], yc[cid]) for cid in south_cell_ids]
    
    # Helper to extract coords from LineString or MultiLineString
    def get_line_coords(geom):
        """Extract coordinates from LineString or MultiLineString."""
        if geom.geom_type == 'LineString':
            return [list(geom.coords)]
        elif geom.geom_type == 'MultiLineString':
            return [list(line.coords) for line in geom.geoms]
        else:
            return []
    
    # Transform boundary segments for context
    north_line_parts = get_line_coords(north_line)
    south_line_parts = get_line_coords(south_line)
    
    north_line_wgs84 = [[transformer.transform(x, y) for x, y in part] for part in north_line_parts]
    south_line_wgs84 = [[transformer.transform(x, y) for x, y in part] for part in south_line_parts]
    
    # Calculate map center (average of all WEL cells)
    all_wel_x = [xc[cid] for cid in north_cell_ids + south_cell_ids]
    all_wel_y = [yc[cid] for cid in north_cell_ids + south_cell_ids]
    center_x = np.mean(all_wel_x)
    center_y = np.mean(all_wel_y)
    center_lon, center_lat = transformer.transform(center_x, center_y)
    
    # Create Folium map
    m = folium.Map(location=[center_lat, center_lon], zoom_start=14, tiles='OpenStreetMap')
    
    # Add north boundary segments
    for part_wgs84 in north_line_wgs84:
        folium.PolyLine(
            locations=[(lat, lon) for lon, lat in part_wgs84],
            color='blue', weight=4, opacity=0.8,
            popup='North boundary segment'
        ).add_to(m)
    
    # Add south boundary segments
    for part_wgs84 in south_line_wgs84:
        folium.PolyLine(
            locations=[(lat, lon) for lon, lat in part_wgs84],
            color='red', weight=4, opacity=0.8,
            popup='South boundary segment'
        ).add_to(m)
    
    # Add north WEL cells as circle markers
    for i, (lon, lat) in enumerate(north_coords_wgs84):
        cell_id = north_cell_ids[i]
        folium.CircleMarker(
            location=[lat, lon],
            radius=5,
            color='darkblue',
            fill=True,
            fillColor='blue',
            fillOpacity=0.7,
            popup=f'North WEL cell {cell_id}<br>Q = {q_per_cell_north:.1f} m³/day'
        ).add_to(m)
    
    # Add south WEL cells as circle markers
    for i, (lon, lat) in enumerate(south_coords_wgs84):
        cell_id = south_cell_ids[i]
        folium.CircleMarker(
            location=[lat, lon],
            radius=5,
            color='darkred',
            fill=True,
            fillColor='red',
            fillOpacity=0.7,
            popup=f'South WEL cell {cell_id}<br>Q = {q_per_cell_south:.1f} m³/day'
        ).add_to(m)
    
    # Add legend
    legend_html = """
    <div style="position: fixed; bottom: 50px; left: 50px; z-index: 1000; 
                background-color: white; padding: 10px; border: 2px solid grey; 
                border-radius: 5px; font-size: 12px;">
        <b>WEL Lateral Inflow</b><br>
        <i style="background:blue; width:12px; height:12px; display:inline-block; border-radius:50%;"></i> North: """ + str(len(north_cell_ids)) + """ cells<br>
        <i style="background:red; width:12px; height:12px; display:inline-block; border-radius:50%;"></i> South: """ + str(len(south_cell_ids)) + """ cells<br>
        <hr style="margin:5px 0;">
        Total: """ + str(len(wel_spd)) + """ cells
    </div>
    """
    m.get_root().html.add_child(folium.Element(legend_html))
    
    # Display map
    display(m)
    
    # Print summary
    print(f"\nWEL cells displayed on interactive map:")
    print(f"  North boundary: {len(north_cell_ids)} cells (blue)")
    print(f"  South boundary: {len(south_cell_ids)} cells (red)")
    print(f"  Click on cells to see details. Zoom/pan to inspect boundaries.")
else:
    print("No WEL cells to visualize")

### 5.5 Areal Recharge (RCH)

Net areal recharge represents the water that infiltrates through the land surface and reaches the water table. In the perceptual model (Notebook 2), we estimated the net recharge for the Limmat Valley to be approximately **110 mm/year**.

This value accounts for:
- Precipitation (~1100 mm/year)
- Evapotranspiration losses
- Surface runoff
- Urbanization effects (reduced infiltration in built-up areas)

The net recharge of ~10% of annual precipitation is typical for urbanized alluvial valleys in temperate climates.

In [ ]:
# Recharge calculation
# Net recharge rate from Notebook 2 perceptual model analysis
# Value for Limmat Valley: ~110 mm/year (10% of precipitation)

annual_recharge_mm = 110  # mm/year (net recharge from perceptual model)
recharge_m_per_day = annual_recharge_mm / 1000 / 365.25  # Convert to m/day

print(f"Annual net recharge: {annual_recharge_mm} mm/year")
print(f"  (~{annual_recharge_mm/1100*100:.0f}% of annual precipitation)")
print(f"Daily recharge rate: {recharge_m_per_day:.2e} m/day")

# Apply uniform recharge to all active cells
recharge_array = np.where(idomain[0] > 0, recharge_m_per_day, 0.0)

In [ ]:
# Calculate total recharge flux for checkpoint
avg_cell_area = boundary_gdf.geometry.area.sum() / ncpl  # m²
total_recharge_m3_day = recharge_m_per_day * avg_cell_area * n_active

# Annual volume for comparison with perceptual model
total_recharge_m3_year = total_recharge_m3_day * 365.25

print(f"Total recharge flux: {total_recharge_m3_day:.0f} m³/day")
print(f"                   = {total_recharge_m3_year/1e6:.2f} million m³/year")

### 5.6 Groundwater Pumping and Thermal Use (WEL)

Beyond natural boundary conditions, the Limmat Valley aquifer is actively used for **drinking water supply** and **thermal energy**. In MODFLOW, both pumping (extraction) and injection wells are represented using the **WEL package**.

#### Hardhof Well Field (Not Modeled)

The northern part of the mid-stream portion of the Limmat Valley hosts the **Hardhof groundwater works** which is an integral part of Zurich's drinking water production infrastructure. Hardhof uses a sophisticated system of:

- **Artificial recharge basins**: River water is spread over infiltration areas to enhance groundwater recharge
- **River bank filtration**: Natural infiltration from the Limmat induced by pumping
- **Vertical injection wells**: Conventional infiltration wells
- **Horizontal collector wells**: Radial wells that draw water from a larger aquifer volume

**Why we exclude Hardhof from our model:** The artificial recharge at Hardhof largely compensates for groundwater extraction. This means the net effect on regional groundwater flow is minimal – water infiltrated through recharge basins replaces the water pumped from wells. As a result, groundwater from upstream continues to flow parallel to the main flow direction (westward along the valley) rather than being diverted toward the Hardhof wells. This allows us to model the upstream aquifer system without explicitly representing the complex Hardhof infrastructure. It is a simplification but allows us to focus on the natural flow system and thermal use scenarios in the upstream and central portions of the valley without overcomplicating this course.

#### Thermal Groundwater Use (Focus of Case Studies)

In the upstream and central portions of the model domain, groundwater is increasingly used for **heating and cooling** buildings via **groundwater heat pump systems (GWHP)**. These systems:

- **Extract** groundwater from a pumping well
- **Transfer heat** to/from the water using a heat exchanger
- **Re-inject** the thermally altered water into an injection well

| System Type | Extraction | Injection | Thermal Effect |
|-------------|------------|-----------|----------------|
| **Heating mode** | Warmer GW | Cooler return | Cooling plume |
| **Cooling mode** | Cooler GW | Warmer return | Warming plume |

In MODFLOW, these doublet systems are represented as:
- **Extraction well**: Negative Q (removes water)
- **Injection well**: Positive Q (adds water, same magnitude)

The **net effect on the water balance is zero** (what goes out comes back in), but the **local flow field** is significantly altered – creating capture zones around extraction wells and mounding around injection wells.

> **📚 Your Case Study Focus**
>
> In your project work, you will add heat pump well systems to this base model to investigate:
> - How does pumping affect local groundwater levels?
> - What is the extent of the capture zone?
> - How do multiple systems interact?
>
> The calibrated model from Notebook 5 will serve as your starting point for these scenario analyses.

#### Why No Wells in the Base Model?

This base model represents **natural conditions** without anthropogenic pumping or injection. This allows us to:
1. Calibrate the model against historical observations
2. Establish a baseline for comparison
3. Add wells as scenarios in case study work

Students will add their own well configurations in the project notebooks, using the WEL package syntax shown in section 5.4 (lateral inflow).

### 5.7 - Numerical Checkpoint 3: Water Balance

In [ ]:
check_task_with_solution('task04_checkpoint_3')

---

All boundary conditions are now defined. We can assemble the MODFLOW 6 packages and run the simulation. The model will solve for steady-state heads that balance all fluxes.

## 6 - Build and Run Model

In [ ]:
# Create the groundwater flow model with Newton-Raphson formulation
# Newton is essential for robust unconfined flow simulation
gwf = ModflowGwf(
    sim, 
    modelname=sim_name, 
    save_flows=True,
    newtonoptions='NEWTON UNDER_RELAXATION'  # Enable Newton with damping
)
print("GWF model created with Newton-Raphson formulation")

In [ ]:
# Create DISV (Discretization by Vertices) package
disv = ModflowGwfdisv(
    gwf, nlay=1, ncpl=ncpl,
    nvert=gridprops['nvert'],
    vertices=gridprops['vertices'],
    cell2d=gridprops['cell2d'],
    top=model_top, 
    botm=[model_bottom],
    idomain=idomain
)
print(f"DISV package created: {ncpl} cells, 1 layer")

In [ ]:
# Node Property Flow (NPF) package - hydraulic properties
npf = ModflowGwfnpf(gwf, icelltype=1, k=k_array, save_flows=True)
print(f"NPF package created with K = {k_array.mean():.1f} m/day")

In [ ]:
# Initial Conditions (IC) package
# Set initial heads to 2 meters below land surface
# This provides a reasonable starting point for the Newton solver

initial_head_depth = 2.0  # meters below land surface

initial_heads = model_top - initial_head_depth

# Ensure initial heads are above aquifer bottom
initial_heads = np.maximum(initial_heads, model_bottom + 0.5)

ic = ModflowGwfic(gwf, strt=initial_heads)
print(f"IC package created:")
print(f"  Initial heads: {initial_head_depth} m below land surface")
print(f"  Head range: {initial_heads.min():.1f} to {initial_heads.max():.1f} m a.s.l.")

In [ ]:
# Constant Head (CHD) package - downstream boundary
chd = ModflowGwfchd(gwf, stress_period_data=chd_spd)
print(f"CHD package created with {len(chd_spd)} boundary cells")

In [ ]:
# Recharge (RCH) package
rcha = ModflowGwfrcha(gwf, recharge=recharge_array)
print(f"RCH package created with rate = {recharge_m_per_day:.2e} m/day")

In [ ]:
# WEL package - lateral inflow from valley margins
if len(wel_spd) > 0:
    wel = ModflowGwfwel(gwf, stress_period_data=wel_spd)
    print(f"WEL package created with {len(wel_spd)} lateral inflow cells")
    print(f"  Total inflow: {sum([w[1] for w in wel_spd]):.0f} m³/day")
else:
    print("No WEL package created (no lateral inflow cells defined)")

In [ ]:
# RIV package - river-aquifer interaction
if len(riv_spd) > 0:
    riv = ModflowGwfriv(gwf, stress_period_data=riv_spd)
    print(f"RIV package created with {len(riv_spd)} river cells")
else:
    print("No RIV package created (no river cells defined)")

In [ ]:
# Iterative Model Solution (IMS) for Newton-Raphson solver
# Newton formulation helps with unconfined aquifer nonlinearity
ims = ModflowIms(
    sim, 
    complexity='MODERATE',
    outer_maximum=500,             # Allow enough iterations
    inner_maximum=100,
    outer_dvclose=1e-3,            # Head change tolerance (1 mm)
    inner_dvclose=1e-4,
    linear_acceleration='BICGSTAB',
    # Newton-specific parameters for robust unconfined flow
    under_relaxation='DBD',        # Damped backtracking
    under_relaxation_gamma=0.0,    # Residual change criterion
    under_relaxation_theta=0.97,   # History-dependent relaxation factor
    backtracking_number=5,         # Number of backtracking iterations
)
sim.register_ims_package(ims, [gwf.name])
print("IMS solver configured for Newton-Raphson solution")

In [ ]:
# Output Control (OC) package
oc = ModflowGwfoc(
    gwf,
    head_filerecord=f'{sim_name}.hds',
    budget_filerecord=f'{sim_name}.cbc',
    saverecord=[('HEAD', 'ALL'), ('BUDGET', 'ALL')]
)
print("Output control configured")

In [ ]:
# Write model files
sim.write_simulation()
print(f"Model files written to: {MODEL_WS}")

In [ ]:
# Run the simulation
print("Running MODFLOW 6 simulation...")
success, buff = sim.run_simulation(silent=True)

if success:
    print("Simulation completed successfully!")
else:
    print("Simulation failed. Check the listing file for errors.")
    print(f"Listing file: {os.path.join(MODEL_WS, sim_name + '.lst')}")

---

The model ran successfully - but what do the results tell us? In this section, we examine the simulated heads and water balance to verify the model is behaving as expected.

## 7 - Initial Results

This is an **uncalibrated** model with initial parameter estimates.

In [ ]:
if success:
    # Read head file
    head_file = flopy.utils.HeadFile(os.path.join(MODEL_WS, f'{sim_name}.hds'))
    heads = head_file.get_data()
    head_layer1 = heads[0].flatten()
    
    # Head validation
    # MODFLOW uses special values for inactive/dry cells:
    # - HDRY (typically 1e30): Cell went dry during simulation
    # - HNOFLO (typically 1e30): Inactive cell (idomain <= 0)
    
    hdry_threshold = 1e10  # Values with |head| above this indicate dry/inactive cells
    
    # Identify problematic cells
    inactive_mask = idomain[0].flatten() <= 0
    dry_mask = (np.abs(head_layer1) > hdry_threshold) & ~inactive_mask
    valid_mask = ~inactive_mask & ~dry_mask
    
    n_inactive = inactive_mask.sum()
    n_dry = dry_mask.sum()
    n_valid = valid_mask.sum()
    
    print("Head validation:")
    print(f"  Total cells: {ncpl}")
    print(f"  Inactive cells (idomain <= 0): {n_inactive}")
    print(f"  Dry cells (|head| > {hdry_threshold:.0e}): {n_dry}")
    print(f"  Valid cells with computed heads: {n_valid}")
    
    if n_dry > 0:
        print(f"\n  ⚠️ WARNING: {n_dry} cells went dry during simulation!")
        print(f"     This may indicate:")
        print(f"     - Aquifer bottom too high relative to boundary conditions")
        print(f"     - Insufficient recharge or inflow")
        print(f"     - Excessive pumping or river leakage")
    
    # Report statistics for valid cells only
    if n_valid > 0:
        valid_heads = head_layer1[valid_mask]
        print(f"\nSimulated head range (valid cells):")
        print(f"  Min: {valid_heads.min():.1f} m a.s.l.")
        print(f"  Max: {valid_heads.max():.1f} m a.s.l.")
        print(f"  Mean: {valid_heads.mean():.1f} m a.s.l.")
    else:
        print("\n  ERROR: No valid head values computed!")
else:
    print("Cannot load results - simulation did not complete.")

In [ ]:
if success:
    # Create masked array for plotting (mask dry/inactive cells)
    head_plot = np.ma.masked_where(
        (np.abs(head_layer1) > hdry_threshold) | (idomain[0].flatten() <= 0),
        head_layer1
    )
    
    fig, ax = plt.subplots(figsize=(12, 10))
    quick_model_plot(
        modelgrid, head_plot, boundary_gdf=boundary_gdf,
        title='Simulated Groundwater Heads (Uncalibrated)',
        cmap='Blues', colorbar_label='Head (m a.s.l.)', ax=ax
    )
    plt.tight_layout()
    plt.show()

### 7.1 - Water Balance

For a steady-state model, inflows and outflows should balance (error < 0.1%).

In [ ]:
# Read and display budget components
if success:
    budget_df = format_budget_summary(gwf, sim)
    if budget_df is not None:
        print("Water Balance Summary:")
        print(budget_df.to_string())
    else:
        print("Budget summary not available.")

### 7.2 - Interpreting the Water Balance

> **🤔 Think about it**
>
> 1. Which flux component dominates? Is this consistent with your perceptual model?
> 2. The RIV package shows both inflow and outflow - what does this indicate about river-aquifer interaction along the Limmat and Sihl?
> 3. How does the CHD outflow compare to your Darcy-based estimate from Notebook 2?
>
> Hint: In Notebook 2, you estimated the western outflow at ~2,000-20,000 m³/day using Darcy's Law.

> **⚠️ Uncertainty: Uncalibrated Results**
>
> These results use initial parameter estimates. The model ran successfully, but **success ≠ accuracy**. Do not use these results for predictions until after calibration (Notebook 5). The actual uncertainty in simulated heads and fluxes will be constrained during calibration against observed groundwater levels.

### 7.3 - Numerical Checkpoint 4: Model Convergence

In [ ]:
check_task_with_solution('task04_checkpoint_4')

### 7.4 - Conceptual Checkpoint 1: River-Aquifer Interaction

In [ ]:
create_multiple_choice('task04_checkpoint_5')

### 7.5 - Conceptual Checkpoint 2: Model Calibration

**Reflection Question:** What observations would you compare to simulated values to assess model quality?

*Your answer:*

(Double-click to edit)

<details>
<summary><b>Click to reveal suggested answer</b></summary>

**Primary observations:**
1. Head measurements from observation wells
2. River discharge (baseflow) at gauging stations
3. Spring discharge data

The Limmat Valley has 113,710 head measurements from 1974-2024 available for calibration in Notebook 5.
</details>

---

## 8 - Summary

### What You're Taking Forward

From this model implementation, you now have:

| For Notebook 5 (Calibration) | Value/Insight |
|------------------------------|---------------|
| Working MODFLOW 6 model | River-aligned DISV grid with complete BCs |
| Dominant flux | River-aquifer interaction (RIV package) |
| Model area | ~10.4 km² with 18,785 cells |
| Initial K estimate | 20 m/day (uniform, needs zone-based refinement) |
| Recharge | ~110 mm/year (3.0×10⁻⁴ m/day) |
| Key calibration target | K values by zone, leakage coefficients |

**Key takeaways:**
- MODFLOW 6 with Newton-Raphson solver handles unconfined flow robustly
- **River-aligned grids** ensure accurate river-aquifer exchange calculations  
- DISV grids provide flexibility for local refinement in case study work
- The model is **uncalibrated** - do not use for predictions yet

**The big picture:** We have a working numerical model that formalizes your perceptual understanding. The next step is to optimize parameters against observed groundwater levels.

---

**Continue to:** [5_calibration.ipynb](5_calibration.ipynb) – Calibrate model parameters against observed heads

In [ ]:
print(f"Model saved to: {MODEL_WS}")
print("This simulation will be the starting point for calibration in Notebook 5.")

---

**Navigation:** [< Notebook 3: MODFLOW Fundamentals](3_modflow_fundamentals.ipynb) | [Notebook 5: Calibration >](5_calibration.ipynb)

---

## 9 - Optional: Additional Details

### 9.1 - DISV Cell Numbering

DISV grids use 1D cell indexing (0 to ncpl-1). The `cell2d` array defines cell ID, center coordinates, and vertex connectivity.

In [ ]:
print("Example cell2d entries (first 3 cells):")
for i, cell in enumerate(gridprops['cell2d'][:3]):
    print(f"  Cell {cell[0]}: center=({cell[1]:.1f}, {cell[2]:.1f}), {cell[3]} vertices")

### 9.2 - Export Grid to GeoPackage

In [ ]:
if success:
    # Use masked heads for export (replace dry cells with NaN)
    head_export = np.where(
        (np.abs(head_layer1) > hdry_threshold) | (idomain[0].flatten() <= 0),
        np.nan,
        head_layer1
    )
    
    output_gpkg = os.path.join(MODEL_WS, 'model_results.gpkg')
    export_grid_to_geopackage(
        modelgrid, head_export, output_path=output_gpkg,
        layer_name='simulated_heads', array_name='head_m'
    )
    print(f"Results exported to: {output_gpkg}")

## References

[\[1\]](#2---Model-Grid---DISV) FloPy Documentation: Unstructured Grids and DISV. [https://flopy.readthedocs.io/](https://flopy.readthedocs.io/) (accessed 2025)

[\[2\]](#5---Boundary-Conditions) Doppler, T., Hendricks Franssen, H.-J., Kaiser H.-P., Kuhlman U., Stauffer, F. (2007): Field evidence of a dynamic leakage coefficient for modelling river–aquifer interactions. Journal of Hydrology, Volume 347, Issues 1–2, DOI: https://doi.org/10.1016/j.jhydrol.2007.09.017.

[\[3\]](#5---Boundary-Conditions) MODFLOW 6 User Guide: River Package (RIV). U.S. Geological Survey. [https://www.usgs.gov/software/modflow-6-usgs-modular-hydrologic-model](https://www.usgs.gov/software/modflow-6-usgs-modular-hydrologic-model) (accessed 2025)

[\[4\]](#3---Model-Geometry) Federal Office of Topography swisstopo, swissALTI3D height model (2m resolution). [https://www.swisstopo.admin.ch/en/height-model-swissalti3d](https://www.swisstopo.admin.ch/en/height-model-swissalti3d) (accessed 2025)

[\[5\]](#3---Model-Geometry) GIS-browser of the canton of Zurich: Groundwater thickness data (GS_GW_MAECHTIGKEIT_L). [https://www.gis.zh.ch/](https://www.gis.zh.ch/) (accessed 2025)